# 📊 Prévision des Charges — Yazaki 2026
### Charges Téléphoniques & Charges d'Impression — par Département

**Librairies :** `statsmodels` - `pmdarima` - `prophet` - `sklearn` - `matplotlib`

| Section | Contenu |
|---|---|
| 1 | Imports |
| 2 | Chargement ETL |
| 3 | Preparation des series |
| 4 | Visualisation exploratoire |
| 5 | Analyse statistique (decomposition, ADF, ACF) |
| 5b | Walk-Forward Cross-Validation - Validation robustesse |
| 6 | Fonctions de prevision - 5 modeles |
| 7 | Lancer les previsions |
| 8 | Visualisation : historique + test + prevision |
| 9 | Tableau des performances |
| 9c | Bar plots + Heatmaps avancees |
| 10 | Export CSV |

## 1. Imports

In [1]:
import sys, os, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Modeles de series temporelles
from statsmodels.tsa.holtwinters    import ExponentialSmoothing

# Analyse des series (statsmodels)
from statsmodels.tsa.stattools      import adfuller           # test ADF
from statsmodels.tsa.seasonal       import seasonal_decompose # decomposition
from statsmodels.graphics.tsaplots  import plot_acf           # ACF

# Selection automatique ARIMA/SARIMA
from pmdarima import auto_arima   # pip install pmdarima

# Metriques
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("OK - Toutes les librairies importees")

OK - Toutes les librairies importees


## 2. Chargement des donnees (ETL)

In [2]:
from pathlib import Path

# Rend le notebook portable: local Windows + Airflow Docker
project_root = Path(os.getenv('YAZAKI_PROJECT_ROOT', Path.cwd()))
if not (project_root / 'etl').exists():
    parent = Path.cwd().parent
    if (parent / 'etl').exists():
        project_root = parent

sys.path.insert(0, str(project_root))
from etl.extract import extract_charges_telephoniques, extract_charges_impression
from etl.transform import transform_charges_telephoniques, transform_charges_impression

charges_tel = transform_charges_telephoniques(extract_charges_telephoniques())
charges_imp = transform_charges_impression(extract_charges_impression())

print(f"OK - Project root: {project_root}")
print(f"OK - ChargesTelephoniques : {len(charges_tel):,} lignes")
print(f"OK - ChargesImpression    : {len(charges_imp):,} lignes")

[INFO] Validation dates ChargesTelephoniques - colonne 'DateOperation'...
  ✓ Toutes les dates valides (3768 lignes)
[INFO] Normalisation NomDepartement (ChargesTelephoniques) - propagation dernière valeur valide du mois...


  ✓ 3743 combinaison(s) traité(e)s
  ⚠️  25 ligne(s) avec département invalide → 'INCONNU'
[INFO] Uniformisation NomDepartement (le plus récurrent par CodeEmployee)...
  ✓ 150 CodeEmployee(s) ont un NomDepartement unique
[INFO] Normalisation CodeEmployee (format: YAZ+nombre ou INCONNU)...
  ✓ CodeEmployee reformatés
[INFO] Propagation NomRole et NumeroTelephone (dernier jour du mois → tous les jours du mois)...


  ✓ 3768 combinaison(s) traité(e)s
[INFO] CodeRole: 79 valeur(s) non mappée(s) → 'INCONNU'
[INFO] Suppression doublons ChargesTelephoniques...
  ✓ Aucun doublon détecté
[INFO] Réassignation IDs et tri ChargesTelephoniques...
  ✓ 3768 lignes conservées, IDs réassignés (1-3768)
    - 3768 lignes valides
    - 0 lignes avec date invalide


[INFO] Validation dates ChargesImpression - colonne 'DateImpression'...
  ✓ Toutes les dates valides (5233 lignes)
[INFO] Normalisation NomDepartement (ChargesImpression)...
  ⚠️  55 département(s) invalide(s) → 'INCONNU'
[INFO] Validation TypeImpression...
  ✓ Tous les types impression valides
[INFO] Extraction CouleurImpression et FormatPapier...
  ✓ Extraction complète
[INFO] Correction CoutUnitaire...
  ✓ CoutUnitaire corrigés
[INFO] Correction NbPages (positif, NA → 0)...
  ✓ NbPages corrigés
[INFO] Réassignation IDs et tri ChargesImpression...
  ✓ 5233 lignes conservées, IDs réassignés (1-5233)
    - 5233 lignes valides
    - 0 lignes avec date invalide
OK - Project root: /opt/airflow/project
OK - ChargesTelephoniques : 3,768 lignes
OK - ChargesImpression    : 5,233 lignes


## 3. Preparation des series mensuelles par departement

In [3]:
# Charges Telephoniques
charges_tel['DateOperation']   = pd.to_datetime(charges_tel['DateOperation'])
charges_tel['NomDepartement']  = charges_tel['NomDepartement'].astype(str).str.strip()
charges_tel = charges_tel[charges_tel['NomDepartement'].str.upper() != 'INCONNU']

tel_par_dept = (
    charges_tel
    .assign(Mois=charges_tel['DateOperation'].dt.to_period('M'))
    .groupby(['Mois','NomDepartement'])['ForfaitTND']
    .sum().unstack(fill_value=0).astype(float)
)
tel_par_dept.index = tel_par_dept.index.to_timestamp()

# Charges Impression
charges_imp['DateImpression']  = pd.to_datetime(charges_imp['DateImpression'])
charges_imp['NomDepartement']  = charges_imp['NomDepartement'].astype(str).str.strip()
charges_imp = charges_imp[charges_imp['NomDepartement'].str.upper() != 'INCONNU']
charges_imp['CoutImp'] = charges_imp['NbPages'] * charges_imp['CoutUnitaire']

imp_par_dept = (
    charges_imp
    .assign(Mois=charges_imp['DateImpression'].dt.to_period('M'))
    .groupby(['Mois','NomDepartement'])['CoutImp']
    .sum().unstack(fill_value=0).astype(float)
)
imp_par_dept.index = imp_par_dept.index.to_timestamp()

# Export CSV brut
tel_par_dept.to_csv('tel_par_dept.csv', index_label='Mois')
imp_par_dept.to_csv('imp_par_dept.csv', index_label='Mois')

print(f"OK Tel : {tel_par_dept.shape[1]} dep | {len(tel_par_dept)} mois ({tel_par_dept.index[0].date()} -> {tel_par_dept.index[-1].date()})")
print(f"OK Imp : {imp_par_dept.shape[1]} dep | {len(imp_par_dept)} mois ({imp_par_dept.index[0].date()} -> {imp_par_dept.index[-1].date()})")

OK Tel : 16 dep | 48 mois (2021-12-01 -> 2025-11-01)
OK Imp : 16 dep | 17 mois (2024-08-01 -> 2025-12-01)


## 4. Visualisation exploratoire

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
fig.suptitle("Apercu des charges mensuelles par departement", fontsize=14, fontweight='bold')

for row, (df_src, label, color) in enumerate([
    (tel_par_dept, "Charges Telephoniques (TND)", "steelblue"),
    (imp_par_dept, "Charges Impression (TND)",    "darkorange"),
]):
    # Top 5 departements
    top5 = df_src.sum().nlargest(5).index
    for dept in top5:
        axes[row,0].plot(df_src.index, df_src[dept], marker='o', markersize=3, label=dept)
    axes[row,0].set_title(f"Top 5 departements - {label}")
    axes[row,0].legend(fontsize=8); axes[row,0].set_ylim(bottom=0)
    axes[row,0].grid(True, alpha=0.4); axes[row,0].tick_params(axis='x', rotation=20, labelsize=8)

    # Total mensuel
    total = df_src.sum(axis=1)
    axes[row,1].fill_between(total.index, total.values, alpha=0.25, color=color)
    axes[row,1].plot(total.index, total.values, color=color, linewidth=2)
    axes[row,1].set_title(f"Total mensuel - {label}")
    axes[row,1].set_ylim(bottom=0); axes[row,1].grid(True, alpha=0.4)
    axes[row,1].tick_params(axis='x', rotation=20, labelsize=8)

plt.tight_layout()
plt.show()

## 5. Analyse statistique des series

Pour **justifier le choix des modeles** on utilise 3 outils standards de statsmodels :
- **Decomposition** : separe tendance / saisonnalite / residu
- **Test ADF** : verifie si la serie est stationnaire (important pour ARIMA)
- **ACF** : detecte la saisonnalite et l'autocorrelation (pic a lag 12 = saisonnalite annuelle)

> On analyse les 3 departements les plus charges comme exemple representatif.

In [5]:
def analyser_serie(y, dates, titre, couleur='steelblue', out_dir='analyses'):

    """Analyse statistique d'une serie avec sauvegarde organisee"""

    os.makedirs(out_dir, exist_ok=True)

    serie = pd.Series(y, index=dates)



    fig = plt.figure(figsize=(16, 10))

    fig.suptitle(f"Analyse - {titre}", fontsize=13, fontweight='bold')

    gs  = fig.add_gridspec(3, 3, hspace=0.5, wspace=0.35)



    # -- Graphique 1 : serie brute + tendance (moyenne mobile 6 mois)

    ax1 = fig.add_subplot(gs[0, :2])

    ax1.plot(dates, y, color=couleur, linewidth=1.8, label='Serie brute')

    tendance = serie.rolling(window=6, center=True).mean()

    ax1.plot(dates, tendance, color='red', linewidth=2,

             linestyle='--', label='Tendance (moy. mobile 6m)')

    ax1.set_title('Serie brute + Tendance')

    ax1.legend(fontsize=8); ax1.set_ylim(bottom=0)

    ax1.grid(True, alpha=0.3); ax1.tick_params(axis='x', rotation=20, labelsize=8)



    # -- Graphique 2 : resultat test ADF (texte)

    ax2 = fig.add_subplot(gs[0, 2])

    ax2.axis('off')

    adf    = adfuller(y, autolag='AIC')

    stat   = "OUI" if adf[1] < 0.05 else "NON"

    conseil = "Serie prete pour ARIMA" if adf[1] < 0.05 else "Differenciation d(1) necessaire"

    texte  = (

        f"Test ADF - Stationnarite"

        f"{'_'*26}"

        f"Statistique : {adf[0]:.3f}"

        f"p-value     : {adf[1]:.4f}"

        f"Seuil 5%    : {adf[4]['5%']:.3f}"

        f"{'_'*26}"

        f"Stationnaire : {stat}"

        f"=> {conseil}"

    )

    ax2.text(0.05, 0.95, texte, transform=ax2.transAxes,

             fontsize=9, verticalalignment='top', fontfamily='monospace',

             bbox=dict(boxstyle='round', facecolor='#f0f9ff',

                       edgecolor='#3b82f6', alpha=0.8))



    # -- Graphiques 3-4-5 : decomposition (tendance / saisonnalite / residu)

    try:

        decomp = seasonal_decompose(serie, model='additive',

                                    period=12, extrapolate_trend='freq')

        for ax, data, title, color in [

            (fig.add_subplot(gs[1, 0]), decomp.trend,    'Tendance',     'red'),

            (fig.add_subplot(gs[1, 1]), decomp.seasonal, 'Saisonnalite', 'green'),

            (fig.add_subplot(gs[1, 2]), decomp.resid,    'Residu',       'purple'),

        ]:

            ax.plot(dates, data, color=color, linewidth=1.5)

            if title == 'Residu': ax.axhline(0, color='black', linewidth=0.8)

            ax.set_title(title); ax.grid(True, alpha=0.3)

            ax.tick_params(axis='x', rotation=20, labelsize=7)

    except Exception as e:

        print(f"  Decomposition non disponible : {e}")



    # -- Graphique 6 : ACF (detecte la saisonnalite)

    ax6 = fig.add_subplot(gs[2, :])

    serie_len = len(serie.dropna())

    if serie_len >= 2:

        max_lags = min(24, serie_len - 1)

        plot_acf(serie, lags=max_lags, ax=ax6, color=couleur, alpha=0.05)

        ax6.set_title(f'ACF - Autocorrelogramme (lags={max_lags})')

        if max_lags >= 12:

            ax6.axvline(12, color='red', linestyle=':', linewidth=1.5, label='Lag 12')

            ax6.legend(fontsize=8)

    else:

        ax6.text(0.5, 0.5, 'ACF non disponible: serie trop courte', ha='center', va='center')

        ax6.set_title('ACF indisponible')

    ax6.set_xlabel('Lag (mois)'); ax6.grid(True, alpha=0.3)



    path = os.path.join(out_dir, f"analyse_{titre.replace(' ','_').replace('/','_')}.png")

    plt.savefig(path, dpi=120, bbox_inches='tight')

    plt.show()



# Analyser les 3 departements les plus charges de chaque type

print("-- Charges Telephoniques --")

for dept in tel_par_dept.sum().nlargest(3).index:

    analyser_serie(tel_par_dept[dept].values, tel_par_dept.index, dept, 'steelblue', out_dir='analyses_tel')



print("\n-- Charges Impression --")

for dept in imp_par_dept.sum().nlargest(3).index:

    analyser_serie(imp_par_dept[dept].values, imp_par_dept.index, dept, 'darkorange', out_dir='analyses_imp')



print("\nOK - Fichiers PNG organises dans :")

print("  analyses_tel/")

print("  analyses_imp/")

-- Charges Telephoniques --



-- Charges Impression --
  Decomposition non disponible : x must have 2 complete cycles requires 24 observations. x only has 17 observation(s)


  Decomposition non disponible : x must have 2 complete cycles requires 24 observations. x only has 17 observation(s)


  Decomposition non disponible : x must have 2 complete cycles requires 24 observations. x only has 17 observation(s)



OK - Fichiers PNG organises dans :
  analyses_tel/
  analyses_imp/


## 5b. Walk-Forward Cross-Validation - Validation de robustesse

**Walk-forward** = validation progressive avec fenêtre croissante.
On réentraîne sur différentes tailles de données d'entraînement et teste toujours sur 6 mois.
→ Démontre la **stabilité du modèle** avant de faire les prévisions finales.

In [6]:
def walk_forward_validation(y, label_dept, n_test=6, window_sizes=[18, 21, 24, 27, 30, 33]):
    """Walk-forward validation : fenetre croissante"""
    rmse_results = []
    
    for window_size in window_sizes:
        if len(y) < window_size + n_test:
            continue
        
        y_train = y[:window_size]
        y_test  = y[window_size:window_size + n_test]
        
        # Tester Holt-Winters
        try:
            m = ExponentialSmoothing(y_train, trend='add', seasonal='add',
                                     seasonal_periods=12,
                                     initialization_method='estimated').fit(optimized=True)
            pred = np.clip(m.forecast(n_test), 0, None)
            rmse = np.sqrt(mean_squared_error(y_test, pred))
            rmse_results.append(rmse)
        except:
            rmse_results.append(np.nan)
    
    return window_sizes[:len(rmse_results)], rmse_results

# Walk-forward pour top 3 depts de chaque type
print("\n" + "="*65)
print("  WALK-FORWARD VALIDATION - Robustesse des modeles")
print("="*65)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Walk-Forward Learning Curve (Holt-Winters)', fontsize=13, fontweight='bold')

for ax, (df_src, resultats_dict, label, color) in enumerate([
    (tel_par_dept, 'resultats_tel', 'Charges Téléphoniques', 'steelblue'),
    (imp_par_dept, 'resultats_imp', 'Charges Impression', 'darkorange'),
]):
    top3_depts = df_src.sum().nlargest(3).index
    
    for dept in top3_depts:
        y = df_src[dept].values.astype(float)
        windows, rmses = walk_forward_validation(y, dept, n_test=6)
        axes[ax].plot(windows, rmses, marker='o', linewidth=2.5, label=dept, alpha=0.8)
    
    axes[ax].set_xlabel('Taille jeu d\'entraînement (mois)', fontsize=11, fontweight='bold')
    axes[ax].set_ylabel('RMSE', fontsize=11, fontweight='bold')
    axes[ax].set_title(label, fontsize=11, fontweight='bold')
    axes[ax].grid(True, alpha=0.3, linestyle='--')
    axes[ax].legend(fontsize=9, loc='best')
    axes[ax].set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('walk_forward_validation.png', dpi=120, bbox_inches='tight')
plt.show()
print("OK - Walk-forward sauvegarde : walk_forward_validation.png")



  WALK-FORWARD VALIDATION - Robustesse des modeles


OK - Walk-forward sauvegarde : walk_forward_validation.png


## 6. Fonctions de prevision - 5 modeles

| Modele | Librairie | Cas d'usage |
|---|---|---|
| **SES** | `statsmodels` | Serie stable, pas de tendance ni saisonnalite |
| **Holt** | `statsmodels` | Tendance sans saisonnalite |
| **Holt-Winters** | `statsmodels` | Tendance + saisonnalite annuelle |
| **ARIMA/SARIMA** | `pmdarima` (auto_arima) | Selection automatique des parametres |
| **Prophet** | `prophet` | Saisonnalite complexe, donnees irregulieres |

Le **meilleur modele** est selectionne par **RMSE minimal** sur les 6 derniers mois.

In [7]:
N_TEST     = 6
N_FORECAST = 12  # Prévisions sur toute l'année 2026 (12 mois)

def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom > 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

def nom_sarima(m):
    p,d,q   = m.order
    P,D,Q,s = m.seasonal_order
    return f"SARIMA({p},{d},{q})({P},{D},{Q})[{s}]" if any([P,D,Q]) else f"ARIMA({p},{d},{q})"

def evaluer_modeles(y, n_test=N_TEST):
    y_train, y_test = y[:-n_test], y[-n_test:]
    resultats = []

    # 1 -- SES
    try:
        m    = ExponentialSmoothing(y_train, trend=None, seasonal=None,
                                    initialization_method='estimated').fit(optimized=True)
        pred = np.clip(m.forecast(n_test), 0, None)
        resultats.append({'nom':'SES', 'type':'ses', 'modele':m, 'pred_test':pred,
                          'rmse': np.sqrt(mean_squared_error(y_test, pred))})
    except: pass

    # 2 -- Holt
    try:
        m    = ExponentialSmoothing(y_train, trend='add', seasonal=None,
                                    initialization_method='estimated').fit(optimized=True)
        pred = np.clip(m.forecast(n_test), 0, None)
        resultats.append({'nom':'Holt', 'type':'holt', 'modele':m, 'pred_test':pred,
                          'rmse': np.sqrt(mean_squared_error(y_test, pred))})
    except: pass

    # 3 -- Holt-Winters
    try:
        m    = ExponentialSmoothing(y_train, trend='add', seasonal='add',
                                    seasonal_periods=12,
                                    initialization_method='estimated').fit(optimized=True)
        pred = np.clip(m.forecast(n_test), 0, None)
        resultats.append({'nom':'Holt-Winters', 'type':'hw', 'modele':m, 'pred_test':pred,
                          'rmse': np.sqrt(mean_squared_error(y_test, pred))})
    except: pass

    # 4 -- ARIMA / SARIMA automatique
    try:
        m    = auto_arima(y_train, seasonal=True, m=12, stepwise=True,
                          suppress_warnings=True, error_action='ignore',
                          max_p=3, max_q=3, max_P=2, max_Q=2, max_d=2, max_D=1)
        pred = np.clip(m.predict(n_periods=n_test), 0, None)
        resultats.append({'nom': nom_sarima(m), 'type':'arima', 'modele':m, 'pred_test':pred,
                          'rmse': np.sqrt(mean_squared_error(y_test, pred))})
    except: pass

    # 5 -- Prophet
    try:
        from prophet import Prophet as Proph
        df_p = pd.DataFrame({'ds': pd.date_range(end=pd.Timestamp.today(),
                              periods=len(y_train), freq='MS'), 'y': y_train})
        m    = Proph(yearly_seasonality=True, weekly_seasonality=False,
                     daily_seasonality=False, seasonality_mode='additive')
        m.fit(df_p)
        fc   = m.predict(m.make_future_dataframe(periods=n_test, freq='MS'))
        pred = np.clip(fc['yhat'].values[-n_test:], 0, None)
        resultats.append({'nom':'Prophet', 'type':'prophet', 'modele':m, 'pred_test':pred,
                          'rmse': np.sqrt(mean_squared_error(y_test, pred))})
    except: pass

    if not resultats:
        return None

    meilleur           = min(resultats, key=lambda r: r['rmse'])
    meilleur['y_test'] = y_test
    meilleur['tous']   = resultats
    return meilleur


def prevoir_departement(nom_dept, y, dates, n_forecast=N_FORECAST, n_test=N_TEST):
    res = evaluer_modeles(y, n_test)
    if res is None:
        print(f"  [{nom_dept}] aucun modele n'a converge")
        return None

    mtype = res['type']
    nom   = res['nom']

    # Reentrainer le meilleur modele sur TOUTE la serie
    try:
        if mtype in ('ses','holt','hw'):
            kw = {'initialization_method':'estimated'}
            if   mtype == 'ses' : kw.update({'trend':None,  'seasonal':None})
            elif mtype == 'holt': kw.update({'trend':'add',  'seasonal':None})
            else                 : kw.update({'trend':'add',  'seasonal':'add', 'seasonal_periods':12})
            mf       = ExponentialSmoothing(y, **kw).fit(optimized=True)
            forecast = np.clip(mf.forecast(n_forecast), 0, None)
            fitted   = mf.fittedvalues.values

        elif mtype == 'arima':
            mf       = auto_arima(y, seasonal=True, m=12, stepwise=True,
                                   suppress_warnings=True, error_action='ignore',
                                   max_p=3, max_q=3, max_P=2, max_Q=2)
            nom      = nom_sarima(mf)
            forecast = np.clip(mf.predict(n_periods=n_forecast), 0, None)
            fitted   = y - mf.resid()

        else:  # prophet
            from prophet import Prophet as Proph
            df_p = pd.DataFrame({'ds': dates, 'y': y})
            mf   = Proph(yearly_seasonality=True, weekly_seasonality=False,
                         daily_seasonality=False, seasonality_mode='additive')
            mf.fit(df_p)
            fc       = mf.predict(mf.make_future_dataframe(periods=n_forecast, freq='MS'))
            forecast = np.clip(fc['yhat'].values[-n_forecast:], 0, None)
            fitted   = fc['yhat'].values[:-n_forecast]
    except:
        forecast = np.full(n_forecast, y[-n_test:].mean())
        fitted   = np.full(len(y), y.mean())

    future_dates = pd.date_range(start=dates[-1] + pd.DateOffset(months=1),
                                  periods=n_forecast, freq='MS')

    rmse_val  = res['rmse']
    mae_val   = mean_absolute_error(res['y_test'], res['pred_test'])
    smape_val = smape(res['y_test'], res['pred_test'])

    comparatif = pd.DataFrame([{
        'Modele'      : r['nom'],
        'RMSE'        : round(r['rmse'], 2),
        'sMAPE%'      : round(smape(res['y_test'], r['pred_test']), 1),
        'Selectionne' : 'OUI' if r['nom'] == nom else '',
    } for r in res['tous']]).sort_values('RMSE').reset_index(drop=True)

    print(f"  OK [{nom_dept:<16}]  {nom:<35}  RMSE={rmse_val:7.1f}  sMAPE={smape_val:.1f}%")

    return {
        'dept'        : nom_dept,
        'modele'      : nom,
        'rmse'        : rmse_val,
        'mae'         : mae_val,
        'smape'       : smape_val,
        'comparatif'  : comparatif,
        'forecast'    : forecast,
        'future_dates': future_dates,
        'fitted'      : fitted,
        'pred_test'   : res['pred_test'],
        'y_test'      : res['y_test'],
        'dates'       : dates,
        'y'           : y,
    }

print("OK - 5 modeles : SES / Holt / Holt-Winters / ARIMA-SARIMA / Prophet")

OK - 5 modeles : SES / Holt / Holt-Winters / ARIMA-SARIMA / Prophet


## 7. Lancer les previsions

In [8]:
print("=" * 65)
print("  CHARGES TELEPHONIQUES - Prevision par departement")
print("=" * 65)
resultats_tel = {}
for dept in tel_par_dept.columns:
    r = prevoir_departement(dept, tel_par_dept[dept].values.astype(float), tel_par_dept.index)
    if r: resultats_tel[dept] = r
print(f"\n  => {len(resultats_tel)} departement(s) traite(s)")

print()
print("=" * 65)
print("  CHARGES IMPRESSION - Prevision par departement")
print("=" * 65)
resultats_imp = {}
for dept in imp_par_dept.columns:
    r = prevoir_departement(dept, imp_par_dept[dept].values.astype(float), imp_par_dept.index)
    if r: resultats_imp[dept] = r
print(f"\n  => {len(resultats_imp)} departement(s) traite(s)")

  CHARGES TELEPHONIQUES - Prevision par departement


  OK [ACHAT           ]  SES                                  RMSE=    0.0  sMAPE=0.0%


  OK [COSEE           ]  SES                                  RMSE=    0.0  sMAPE=0.0%


  OK [DIRECTION       ]  SES                                  RMSE=   73.0  sMAPE=200.0%


  OK [EHS             ]  SES                                  RMSE=   20.6  sMAPE=16.8%


...

  OK [PRODUCTION A    ]  SES                                  RMSE=    0.0  sMAPE=0.0%


  OK [PRODUCTION B    ]  Holt-Winters                         RMSE=   10.9  sMAPE=1.9%


  OK [QUALITE         ]  SES                                  RMSE=    2.0  sMAPE=0.2%


  OK [RH              ]  SES                                  RMSE=    0.0  sMAPE=0.0%


  OK [TD              ]  Holt-Winters                         RMSE=   29.5  sMAPE=6.3%

  => 16 departement(s) traite(s)

  CHARGES IMPRESSION - Prevision par departement
  OK [ACHAT           ]  Holt                                 RMSE=    2.8  sMAPE=93.9%
  OK [COSEE           ]  SES                                  RMSE=    7.8  sMAPE=72.0%
  OK [DIRECTION       ]  SES                                  RMSE=    9.4  sMAPE=72.0%
  OK [EHS             ]  SES                                  RMSE=    4.9  sMAPE=48.9%
  OK [ENGENIERIE      ]  SES                                  RMSE=   19.7  sMAPE=59.6%
  OK [FINANCE         ]  SES                                  RMSE=    3.5  sMAPE=134.5%
  OK [IT              ]  SES                                  RMSE=    2.5  sMAPE=112.0%
  OK [LOGISTIQUE      ]  SES                                  RMSE=   10.2  sMAPE=34.3%
  OK [NYS             ]  SES                                  RMSE=    5.0  sMAPE=76.4%
  OK [OLS             ]  SES       

## 8. Visualisation - Historique, Test, Prevision

Chaque graphique montre **3 zones** :
- **Historique** (trait plein) - les donnees reelles
- **Test set** (tirete) - les 6 derniers mois : reel vs predit par le modele
- **Prevision** (rouge) - les 6 prochains mois avec bande de confiance +/-1sigma

In [9]:
def visualiser_resultats(resultats, label, c_hist, c_test, c_prev, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    n = len(resultats); cols = 3
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5))
    axes = axes.flatten() if n > 1 else [axes]
    fig.suptitle(f"Previsions - {label}", fontsize=14, fontweight='bold')

    for ax, (dept, r) in zip(axes, resultats.items()):
        dates  = r['dates']
        y      = r['y']
        n_test = len(r['y_test'])
        y_train = y[:-n_test]

        # Zone 1 : Historique
        ax.plot(dates[:-n_test], y_train,
                color=c_hist, linewidth=1.8, label='Historique', zorder=3)

        # Zone 2 : Test set - reel (tirete) + predit (carre)
        ax.plot(dates[-n_test:], r['y_test'],
                color=c_hist, linewidth=1.5, linestyle='--', zorder=3)
        ax.plot(dates[-n_test:], r['pred_test'],
                color=c_test, linewidth=2, marker='s', markersize=5,
                linestyle='--', label='Predit (test)', zorder=4)
        ax.fill_between(dates[-n_test:], r['y_test'], r['pred_test'],
                         alpha=0.15, color=c_test)

        # Zone 3 : Prevision future + bande confiance
        sigma = np.std(y - r['fitted'])
        ax.plot(r['future_dates'], r['forecast'],
                color=c_prev, linewidth=2.5, marker='o', markersize=5,
                label='Prevision 2026', zorder=5)
        ax.fill_between(r['future_dates'],
                         np.clip(r['forecast'] - sigma, 0, None),
                         r['forecast'] + sigma,
                         alpha=0.25, color=c_prev, label='+/-1 sigma')

        # Separateurs train/test et test/prevision
        ax.axvline(dates[-n_test], color='gray',  linestyle=':', linewidth=1.2)
        ax.axvline(dates[-1],      color='black', linestyle=':', linewidth=1.2)

        ax.set_title(
            f"{dept}\n{r['modele']}\nRMSE={r['rmse']:.1f}  sMAPE={r['smape']:.1f}%",
            fontsize=8.5, fontweight='bold'
        )
        ax.set_ylim(bottom=0); ax.grid(True, alpha=0.3)
        ax.tick_params(axis='x', labelsize=7, rotation=25)
        ax.legend(fontsize=7, loc='upper left', ncol=2)

    for ax in axes[len(resultats):]:
        ax.set_visible(False)

    plt.tight_layout()
    path = os.path.join(out_dir, 'previsions.png')
    plt.savefig(path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"OK - Sauvegarde : {path}")

visualiser_resultats(resultats_tel, "Charges Telephoniques",
                     c_hist='steelblue', c_test='orange', c_prev='#dc2626',
                     out_dir='previsions_tel')

visualiser_resultats(resultats_imp, "Charges Impression",
                     c_hist='darkorange', c_test='royalblue', c_prev='#7c3aed',
                     out_dir='previsions_imp')

OK - Sauvegarde : previsions_tel/previsions.png


OK - Sauvegarde : previsions_imp/previsions.png


## 9. Tableau des performances

In [10]:
def tableau_performances(resultats, label):
    print(f"\n{'='*65}")
    print(f"  {label} - Resume global (trie par sMAPE%)")
    print(f"{'='*65}")
    rows = [{'Departement': dept, 'Meilleur Modele': r['modele'],
             'RMSE': round(r['rmse'],2), 'MAE': round(r['mae'],2),
             'sMAPE%': round(r['smape'],1)}
            for dept, r in resultats.items()]
    df = pd.DataFrame(rows).set_index('Departement').sort_values('sMAPE%')
    print(df.to_string())
    print(f"\n  sMAPE moyen : {df['sMAPE%'].mean():.1f}%")

    print(f"\n  Comparatif 5 modeles par departement :")
    for dept, r in resultats.items():
        print(f"\n  [{dept}]")
        print(r['comparatif'].to_string(index=False))
    return df

recap_tel = tableau_performances(resultats_tel, "Charges Telephoniques")
recap_imp = tableau_performances(resultats_imp, "Charges Impression")


  Charges Telephoniques - Resume global (trie par sMAPE%)
             Meilleur Modele   RMSE    MAE  sMAPE%
Departement                                       
ACHAT                    SES   0.00   0.00     0.0
COSEE                    SES   0.00   0.00     0.0
IT                       SES   0.00   0.00     0.0
LOGISTIQUE               SES   0.00   0.00     0.0
PRODUCTION A             SES   0.00   0.00     0.0
RH                       SES   0.00   0.00     0.0
QUALITE                  SES   2.04   0.83     0.2
ENGENIERIE               SES   4.56   2.50     0.4
FINANCE                  SES   4.56   2.50     0.5
OLS             Holt-Winters   2.80   2.52     1.0
PRODUCTION B    Holt-Winters  10.86   6.64     1.9
TD              Holt-Winters  29.45  24.15     6.3
PLPP                     SES  73.85  52.49    12.4
EHS                      SES  20.57  15.27    16.8
NYS                     Holt  58.26  44.25    21.3
DIRECTION                SES  73.03  66.67   200.0

  sMAPE moyen : 16.3%


## 9c. Visualisations Avancées - Bar plots + Heatmaps

In [11]:
# ============== BAR PLOTS : Réel vs Prévisions ==============

def comparaison_reel_prevision(resultats, label, c_reel, c_prev, out_file):
    """Bar plot comparant réel (test) vs prévisions futurs"""
    os.makedirs('forecast_imp' if 'Impression' in label else 'forecast_tel', exist_ok=True)
    
    # Sélectionner top 5 depts
    top5 = sorted(resultats.items(), 
                  key=lambda x: x[1]['y'].sum(), reverse=True)[:5]
    
    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    fig.suptitle(f'{label} - Comparaison Réel vs Prévisions (12 mois)', 
                 fontsize=13, fontweight='bold')
    
    for ax, (dept, r) in zip(axes, top5):
        # Derniers 6 mois (réel - test set)
        x_test = np.arange(6)
        ax.bar(x_test - 0.2, r['y_test'], width=0.4, label='Réel (test)', 
               color=c_reel, alpha=0.8)
        ax.bar(x_test + 0.2, r['pred_test'], width=0.4, label='Prédit (test)', 
               color=c_prev, alpha=0.8)
        
        # Les 12 mois futurs (prévisions)
        x_future = np.arange(6, 6 + len(r['forecast']))  # 6 à 17 (12 mois)
        ax.bar(x_future, r['forecast'], width=0.6, 
               color=c_prev, alpha=0.3, edgecolor=c_prev, linewidth=1.5, label='Forecast (12m)')
        
        # MAPE pour ce département
        mape_dept = round(r['smape'], 1)
        ax.set_title(f'{dept}\n({r["modele"]}, sMAPE={mape_dept}%)', 
                     fontsize=9, fontweight='bold')
        ax.set_ylabel('Charge (TND)', fontsize=9)
        
        # X-axis : 6 mois test + 12 mois prévision = 18 total
        ax.set_xticks(np.arange(0, 18))
        labels = [f'T{i+1}' for i in range(6)] + [f'P{i+1}' for i in range(12)]
        ax.set_xticklabels(labels, fontsize=7, rotation=45)
        
        # Ligne de séparation entre test et prévisions
        ax.axvline(5.5, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
        ax.text(2.5, ax.get_ylim()[1]*0.95, 'TEST', fontsize=8, ha='center', 
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))
        ax.text(11.5, ax.get_ylim()[1]*0.95, 'FORECAST', fontsize=8, ha='center',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
        
        ax.grid(True, alpha=0.2, axis='y')
        ax.legend(fontsize=7, loc='upper left')
    
    plt.tight_layout()
    path = os.path.join('forecast_imp' if 'Impression' in label else 'forecast_tel', 
                        out_file)
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"OK - {path}")

comparaison_reel_prevision(resultats_tel, 'Charges Téléphoniques',
                           c_reel='steelblue', c_prev='#dc2626', 
                           out_file='reel_vs_previsions.png')

comparaison_reel_prevision(resultats_imp, 'Charges Impression',
                           c_reel='darkorange', c_prev='#7c3aed',
                           out_file='reel_vs_previsions.png')

OK - forecast_tel/reel_vs_previsions.png


OK - forecast_imp/reel_vs_previsions.png


## 10. Export des previsions en CSV

In [12]:
def exporter(resultats, fichier):
    rows = []
    for dept, r in resultats.items():
        for d, v in zip(r['dates'], r['y']):
            rows.append({'Departement':dept,'Mois':d.date(),'Type':'Historique',
                         'Charge_TND':round(float(v),2),'Modele':''})
        for d, v in zip(r['future_dates'], r['forecast']):
            rows.append({'Departement':dept,'Mois':d.date(),'Type':'Prevision',
                         'Charge_TND':round(float(v),2),'Modele':r['modele']})
    pd.DataFrame(rows).sort_values(['Departement','Mois']).to_csv(fichier, index=False)
    print(f"OK - {fichier}")

exporter(resultats_tel, 'previsions_telephoniques.csv')
exporter(resultats_imp, 'previsions_impression.csv')
print("\nFichiers generes :")
print("  previsions_telephoniques.csv")
print("  previsions_impression.csv")
print("  previsions_tel/previsions.png")
print("  previsions_imp/previsions.png")

OK - previsions_telephoniques.csv
OK - previsions_impression.csv

Fichiers generes :
  previsions_telephoniques.csv
  previsions_impression.csv
  previsions_tel/previsions.png
  previsions_imp/previsions.png
